# Compute LPIPS Metric

This notebook computes Learned Perceptual Image Patch Similarity (LPIPS) between DuoDiT sample folders and ImageNet reference folders, following the class loop used in `FID.ipynb`.

Research notes:
- LPIPS was introduced in Zhang et al., *The Unreasonable Effectiveness of Deep Features as a Perceptual Metric* (CVPR 2018): https://arxiv.org/abs/1801.03924
- The official implementation is `richzhang/PerceptualSimilarity`: https://github.com/richzhang/PerceptualSimilarity
- The official package expects RGB tensors with shape `N x 3 x H x W`, scaled to `[-1, 1]`.
- LPIPS is a distance: lower means more perceptually similar, higher means more different.
- Report the backbone (`alex`, `squeeze`, or `vgg`), LPIPS version, resize size, and pairing rule with every result.

In [1]:
!pip install -q lpips pandas tqdm


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
from __future__ import annotations

import json
import math
import warnings
from pathlib import Path

import lpips
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageOps
from tqdm.auto import tqdm

## Configuration

The paths mirror `FID.ipynb`. LPIPS is pairwise, not distributional, so this notebook pairs generated and reference images by sorted filename order within each class directory. If your generated images have an explicit one-to-one target image, replace `pair_image_paths` with that mapping.

In [14]:
# Classes evaluated in FID.ipynb.
classes = [972, 973, 974, 975, 976]

# Folder prefixes from FID.ipynb.
#paths
new_dit_img_path = "/home/mahmood/github/DuoDiT/duodit_samples/samples_clstoken_20260104_"
lightning_dit = "/home/mahmood/github/LightningDiT/output/sample_compare/lightningdit_800ep_reported_cfg/targeted_classes_"
dit_img_path = "/home/mahmood/github/DuoDiT/dit_samples/samples_org_20251205_"
imgnet_path = "/home/mahmood/github/DuoDiT/imagenet_train/00"
imgnet_val_path = "/home/mahmood/github/DuoDiT/imagenet_val_organized/"

# Output file.
output_path = "logs/lpips_results_cls_972_to_976_2026_05_28.json"

# LPIPS settings.
LPIPS_NET = "alex"       # Official default; fastest and recommended for forward metric scores.
LPIPS_VERSION = "0.1"    # Current official LPIPS version in richzhang/PerceptualSimilarity.
BATCH_SIZE = 16

# Pillow resize uses (width, height). Set to None only if paired images already share the same size.
RESIZE_TO = (256, 256)

# Use None for all sorted pairs, or an integer for a quick smoke test.
MAX_PAIRS_PER_CLASS = None

# Pair-level scores can be large. Keep False for compact JSON summaries.
SAVE_PER_PAIR = False

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loss_fn = lpips.LPIPS(net=LPIPS_NET, version=LPIPS_VERSION).to(device).eval()
print(f"Using device: {device}")

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/home/mahmood/github/DuoDiT/.venv/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/mahmood/github/DuoDiT/.venv/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /home/mahmood/github/DuoDiT/.venv/lib/python3.13/site-packages/lpips/weights/v0.1/alex.pth
Using device: cuda


## Helpers

Each LPIPS score compares one generated image to one reference image. The helper below sorts files, truncates to the smaller directory length, resizes both images to `RESIZE_TO`, scales RGB values from `[0, 1]` to `[-1, 1]`, and evaluates the metric in batches.

In [16]:
RESAMPLE_BICUBIC = Image.Resampling.BICUBIC if hasattr(Image, "Resampling") else Image.BICUBIC


def list_images(directory: str | Path) -> list[Path]:
    path = Path(directory)
    if not path.exists():
        raise FileNotFoundError(f"Directory does not exist: {path}")

    files = sorted(
        p for p in path.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )
    if not files:
        raise FileNotFoundError(f"No image files found in: {path}")
    return files


def pair_image_paths(
    generated_dir: str | Path,
    reference_dir: str | Path,
    max_pairs: int | None = None,
) -> tuple[list[tuple[Path, Path]], dict]:
    generated = list_images(generated_dir)
    reference = list_images(reference_dir)

    if len(generated) != len(reference):
        warnings.warn(
            f"Directory counts differ for {generated_dir} and {reference_dir}: "
            f"{len(generated)} generated vs {len(reference)} reference. "
            "Using the smaller count after sorted pairing."
        )

    paired_count = min(len(generated), len(reference))
    if max_pairs is not None:
        paired_count = min(paired_count, max_pairs)

    counts = {
        "generated_count": len(generated),
        "reference_count": len(reference),
        "paired_count": paired_count,
    }
    return list(zip(generated[:paired_count], reference[:paired_count])), counts


def image_to_lpips_tensor(path: str | Path, resize_to: tuple[int, int] | None = RESIZE_TO) -> torch.Tensor:
    with Image.open(path) as img:
        img = ImageOps.exif_transpose(img).convert("RGB")
        if resize_to is not None:
            img = img.resize(resize_to, resample=RESAMPLE_BICUBIC)
        array = np.asarray(img, dtype=np.float32) / 255.0

    tensor = torch.from_numpy(array).permute(2, 0, 1)
    return tensor.mul(2.0).sub(1.0)


def iter_batches(items: list, batch_size: int):
    if batch_size < 1:
        raise ValueError("batch_size must be >= 1")
    for start in range(0, len(items), batch_size):
        yield items[start:start + batch_size]

In [17]:
def compute_lpips_for_pairs(
    pairs: list[tuple[Path, Path]],
    batch_size: int = BATCH_SIZE,
    resize_to: tuple[int, int] | None = RESIZE_TO,
) -> tuple[dict, list[dict], list[dict]]:
    records: list[dict] = []
    skipped: list[dict] = []

    total_batches = math.ceil(len(pairs) / batch_size)
    with torch.no_grad():
        for batch_pairs in tqdm(iter_batches(pairs, batch_size), total=total_batches, leave=False):
            generated_tensors = []
            reference_tensors = []
            kept_pairs = []

            for generated_path, reference_path in batch_pairs:
                generated_tensor = image_to_lpips_tensor(generated_path, resize_to=resize_to)
                reference_tensor = image_to_lpips_tensor(reference_path, resize_to=resize_to)

                if generated_tensor.shape != reference_tensor.shape:
                    skipped.append({
                        "generated": str(generated_path),
                        "reference": str(reference_path),
                        "generated_shape": tuple(generated_tensor.shape),
                        "reference_shape": tuple(reference_tensor.shape),
                    })
                    continue

                generated_tensors.append(generated_tensor)
                reference_tensors.append(reference_tensor)
                kept_pairs.append((generated_path, reference_path))

            if not generated_tensors:
                continue

            generated_batch = torch.stack(generated_tensors).to(device)
            reference_batch = torch.stack(reference_tensors).to(device)
            batch_scores = loss_fn(generated_batch, reference_batch).flatten().detach().cpu().tolist()

            for (generated_path, reference_path), score in zip(kept_pairs, batch_scores):
                records.append({
                    "generated": generated_path.name,
                    "reference": reference_path.name,
                    "lpips": float(score),
                })

    if not records:
        raise ValueError("No LPIPS scores were computed. Check paths, image sizes, and RESIZE_TO.")

    values = np.array([row["lpips"] for row in records], dtype=np.float64)
    summary = {
        "num_pairs": int(values.size),
        "num_skipped_shape_mismatch": len(skipped),
        "mean": float(values.mean()),
        "std": float(values.std(ddof=1)) if values.size > 1 else 0.0,
        "median": float(np.median(values)),
        "min": float(values.min()),
        "max": float(values.max()),
    }
    return summary, records, skipped

## Compute LPIPS

The output mirrors the FID notebook structure: one result entry per class, with `new_dit` and `org_dit` summaries. Lower LPIPS is better.

In [18]:
model_dirs = {
    "new_dit": new_dit_img_path,
    "org_dit": dit_img_path,
    "lightning_dit": lightning_dit
}

lpips_results = []
pair_records = {}

for cls in classes:
    cls = str(cls)
    reference_dir = imgnet_path + cls # Source of domain data (imgnet_path or imgnet_val_path)
    class_result = {}

    for model_name, generated_prefix in model_dirs.items():
        generated_dir = generated_prefix + cls
        pairs, counts = pair_image_paths(
            generated_dir,
            reference_dir,
            max_pairs=MAX_PAIRS_PER_CLASS,
        )

        print(f"class {cls} | {model_name}: evaluating {counts['paired_count']} pairs")
        summary, records, skipped = compute_lpips_for_pairs(pairs)
        summary.update(counts)
        summary.update({
            "generated_dir": generated_dir,
            "reference_dir": reference_dir,
        })
        class_result[model_name] = summary

        if SAVE_PER_PAIR:
            pair_records.setdefault(cls, {})[model_name] = records

        print(
            f"class {cls} | {model_name}: "
            f"mean LPIPS={summary['mean']:.6f}, "
            f"median={summary['median']:.6f}, "
            f"n={summary['num_pairs']}, "
            f"skipped={summary['num_skipped_shape_mismatch']}"
        )

    lpips_results.append({cls: class_result})

class 972 | new_dit: evaluating 10 pairs


/tmp/ipykernel_101666/4188757466.py:27: UserWarning: Directory counts differ for /home/mahmood/github/DuoDiT/duodit_samples/samples_clstoken_20260104_972 and /home/mahmood/github/DuoDiT/imagenet_train/00972: 10 generated vs 680 reference. Using the smaller count after sorted pairing.
  warnings.warn(


  0%|          | 0/1 [00:00<?, ?it/s]

class 972 | new_dit: mean LPIPS=0.760558, median=0.770949, n=10, skipped=0
class 972 | org_dit: evaluating 10 pairs


/tmp/ipykernel_101666/4188757466.py:27: UserWarning: Directory counts differ for /home/mahmood/github/DuoDiT/dit_samples/samples_org_20251205_972 and /home/mahmood/github/DuoDiT/imagenet_train/00972: 10 generated vs 680 reference. Using the smaller count after sorted pairing.
  warnings.warn(


...

  0%|          | 0/1 [00:00<?, ?it/s]

class 976 | org_dit: mean LPIPS=0.695997, median=0.705656, n=10, skipped=0
class 976 | lightning_dit: evaluating 10 pairs


/tmp/ipykernel_101666/4188757466.py:27: UserWarning: Directory counts differ for /home/mahmood/github/LightningDiT/output/sample_compare/lightningdit_800ep_reported_cfg/targeted_classes_976 and /home/mahmood/github/DuoDiT/imagenet_train/00976: 10 generated vs 644 reference. Using the smaller count after sorted pairing.
  warnings.warn(


  0%|          | 0/1 [00:00<?, ?it/s]

class 976 | lightning_dit: mean LPIPS=0.730105, median=0.739623, n=10, skipped=0


In [19]:
rows = []
for class_entry in lpips_results:
    cls, result_by_model = next(iter(class_entry.items()))
    for model_name, summary in result_by_model.items():
        rows.append({
            "class": cls,
            "model": model_name,
            "mean_lpips": summary["mean"],
            "median_lpips": summary["median"],
            "std_lpips": summary["std"],
            "min_lpips": summary["min"],
            "max_lpips": summary["max"],
            "num_pairs": summary["num_pairs"],
            "skipped": summary["num_skipped_shape_mismatch"],
        })

lpips_df = pd.DataFrame(rows)
display(lpips_df)

mean_by_class = lpips_df.pivot(index="class", columns="model", values="mean_lpips")
display(mean_by_class)

overall = (
    lpips_df.assign(weighted_sum=lpips_df["mean_lpips"] * lpips_df["num_pairs"])
    .groupby("model")
    .agg(
        weighted_sum=("weighted_sum", "sum"),
        num_pairs=("num_pairs", "sum"),
        mean_of_class_means=("mean_lpips", "mean"),
    )
)
overall["weighted_mean_lpips"] = overall["weighted_sum"] / overall["num_pairs"]
display(overall[["weighted_mean_lpips", "mean_of_class_means", "num_pairs"]])

,class,model,mean_lpips,median_lpips,std_lpips,min_lpips,max_lpips,num_pairs,skipped
0,972,new_dit,0.760558,0.770949,0.076929,0.632528,0.863298,10,0
1,972,org_dit,0.740619,0.764655,0.084487,0.583266,0.857134,10,0
2,972,lightning_dit,0.717542,0.715223,0.051304,0.637504,0.793037,10,0
3,973,new_dit,0.700283,0.709237,0.144603,0.473308,0.886649,10,0
4,973,org_dit,0.743557,0.681918,0.132127,0.621821,1.028105,10,0
5,973,lightning_dit,0.761987,0.746744,0.113075,0.572629,0.937231,10,0
6,974,new_dit,0.705049,0.729200,0.062991,0.555186,0.766031,10,0
7,974,org_dit,0.714343,0.694206,0.059406,0.628596,0.812424,10,0
8,974,lightning_dit,0.692179,0.707166,0.065810,0.576613,0.784486,10,0
9,975,new_dit,0.728352,0.727422,0.052231,0.662244,0.837657,10,0


model,lightning_dit,new_dit,org_dit
class,,,
972,0.717542,0.760558,0.740619
973,0.761987,0.700283,0.743557
974,0.692179,0.705049,0.714343
975,0.730957,0.728352,0.733083
976,0.730105,0.692063,0.695997


,weighted_mean_lpips,mean_of_class_means,num_pairs
model,,,
lightning_dit,0.726554,0.726554,50
new_dit,0.717261,0.717261,50
org_dit,0.725520,0.725520,50


In [20]:
payload = {
    "metric": "LPIPS",
    "lower_is_better": True,
    "lpips_net": LPIPS_NET,
    "lpips_version": LPIPS_VERSION,
    "resize_to": RESIZE_TO,
    "pairing_rule": "sorted filenames within each class directory",
    "max_pairs_per_class": MAX_PAIRS_PER_CLASS,
    "sources": [
        "https://arxiv.org/abs/1801.03924",
        "https://github.com/richzhang/PerceptualSimilarity",
    ],
    "results": lpips_results,
}

if SAVE_PER_PAIR:
    payload["pair_records"] = pair_records

output_path = Path(output_path)
output_path.parent.mkdir(parents=True, exist_ok=True)
with output_path.open("w") as fp:
    json.dump(payload, fp, indent=2)

print(f"Saved LPIPS results to {output_path}")

Saved LPIPS results to logs/lpips_results_cls_972_to_976_2026_05_28.json


## Interpretation

Use the mean or weighted mean LPIPS to compare `new_dit` and `org_dit`; smaller values indicate generated images are perceptually closer to the paired references. Because LPIPS is a full-reference metric, the pairing rule matters. If these are unpaired generative samples, treat the result as a sorted-pair proxy and report FID/KID alongside it.